# Table of Contents
- [Import Libraries](#import-libraries)
- [Data Paths](#data-paths)
- [Loading The Data](#loading-the-data)
- [Keeping only Mobile connections](#mobile-only)
- [Keeping all radio generations](#keep-all-generations)
- [Convert Timestamp to date-time in local Riyadh time](#convert-timestamp)
- [Removing duplicated samples](#dedupe-samples)
- [Checking TrafficVolume outliers](#outliers)
- [Total traffic by Operator and Direction (GB)](#traffic-by-operator)
- [Traffic by Hour by Direction](#traffic-by-hour)
- [Saving the cleaned data](#saving)


<a id="import-libraries"></a>
### Import Libraries

In [1]:
import pandas as pd

<a id="data-paths"></a>
### Data Paths

In [2]:
traffic_path = '../TrafficVolume.csv'

<a id="loading-the-data"></a>
### Loading The Data

In [3]:
traffic_df = pd.read_csv(traffic_path)

In [ ]:
traffic_df.head(5)

In [5]:
traffic_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 138469 entries, 0 to 138468
Data columns (total 10 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   Timestamp               138469 non-null  object 
 1   LocationLatitude        138469 non-null  float64
 2   LocationLongitude       138469 non-null  float64
 3   RadioConnectionType     138469 non-null  object 
 4   Country                 138469 non-null  object 
 5   RadioNetworkGeneration  138469 non-null  object 
 6   RadioOperatorName       138469 non-null  object 
 7   TrafficDirection        138469 non-null  object 
 8   TrafficVolume           138469 non-null  float64
 9   RadioMobileDataEnabled  138469 non-null  object 
dtypes: float64(3), object(7)
memory usage: 10.6+ MB


- `Checking Unique Values and Counts`

In [6]:
check_unique = [
 'RadioConnectionType',
 'Country',
 'RadioNetworkGeneration',
 'RadioOperatorName',
 'TrafficDirection',
 'RadioMobileDataEnabled']

for col in check_unique:
    print(f"Value counts in {col}:")
    print(traffic_df[col].value_counts())
    print("-----------------------------")

Value counts in RadioConnectionType:
RadioConnectionType
Mobile    136425
WiFi        2044
Name: count, dtype: int64
-----------------------------
Value counts in Country:
Country
Saudi Arabia    138469
Name: count, dtype: int64
-----------------------------
Value counts in RadioNetworkGeneration:
RadioNetworkGeneration
4G         104208
3G          32707
2G           1533
WiFi           19
Unknown         2
Name: count, dtype: int64
-----------------------------
Value counts in RadioOperatorName:
RadioOperatorName
Operator A    68211
Operator B    40312
Operator C    29946
Name: count, dtype: int64
-----------------------------
Value counts in TrafficDirection:
TrafficDirection
Downlink    69262
Uplink      69207
Name: count, dtype: int64
-----------------------------
Value counts in RadioMobileDataEnabled:
RadioMobileDataEnabled
Enabled     138434
Disabled        35
Name: count, dtype: int64
-----------------------------


- `Checking Nulls`

In [7]:
traffic_df.isna().sum()

Timestamp                 0
LocationLatitude          0
LocationLongitude         0
RadioConnectionType       0
Country                   0
RadioNetworkGeneration    0
RadioOperatorName         0
TrafficDirection          0
TrafficVolume             0
RadioMobileDataEnabled    0
dtype: int64

- `Checking Duplicates`

In [8]:
print(traffic_df.duplicated().sum())

30


- `Dropping Duplicates` and `Country` column since the data is from One country `Saudi Arabia`

In [9]:
traffic_df.drop_duplicates(inplace=True)

In [10]:
traffic_df.drop(columns=['Country'], inplace=True)

<a id="mobile-only"></a>
### Keeping only `Mobile` connections
WiFi and Unknown samples are not carried by the operator's cellular network.

In [11]:
traffic_df = traffic_df[traffic_df['RadioConnectionType'] == 'Mobile']
traffic_df['RadioConnectionType'].value_counts()

RadioConnectionType
Mobile    136395
Name: count, dtype: int64

<a id="keep-all-generations"></a>
### Keeping all radio generations
Unlike the RSRP file, we do **not** filter to `4G` here.
The task asks for traffic *per operator*, and 2G/3G is ~25% of this file,
so dropping it would understate every operator's real traffic.

In [12]:
traffic_df['RadioNetworkGeneration'].value_counts(normalize=True).round(4)

RadioNetworkGeneration
4G    0.7519
3G    0.2370
2G    0.0110
Name: proportion, dtype: float64

<a id="convert-timestamp"></a>
### Convert `Timestamp` to date-time in local Riyadh time
The raw strings carry mixed UTC offsets, so `utc=True` is required to parse them.
A few rows are truncated and have no offset at all, so `errors='coerce'` turns
those into `NaT` instead of raising, and we drop them afterwards.

In [13]:
traffic_df['Timestamp'] = (pd.to_datetime(traffic_df['Timestamp'], utc=True, errors='coerce')
                             .dt.tz_convert('Asia/Riyadh'))

print(traffic_df['Timestamp'].isna().sum())

80


In [14]:
traffic_df = traffic_df[traffic_df['Timestamp'].notna()]
print(traffic_df['Timestamp'].min(), '->', traffic_df['Timestamp'].max())

2019-11-01 21:00:00+03:00 -> 2019-11-05 00:45:00+03:00


<a id="dedupe-samples"></a>
### Removing duplicated samples
`TrafficDirection` has to be part of the key, otherwise the `Uplink` row that
pairs with each `Downlink` row at the same timestamp and location gets deleted.

In [15]:
key = ['Timestamp', 'LocationLatitude', 'LocationLongitude',
       'RadioOperatorName', 'TrafficDirection']

print(len(traffic_df))
traffic_df = traffic_df.drop_duplicates(subset=key)
print(len(traffic_df))

136315
129820


<a id="outliers"></a>
### Checking `TrafficVolume` outliers
The tail is extreme, but these samples are kept: the bubble map sums traffic per
area, and a genuinely huge session is exactly the hotspot the chart should show.
Capping it would hide the answer.

In [16]:
traffic_df.groupby('TrafficDirection')['TrafficVolume'].describe(percentiles=[.5, .9, .99, .999])

,count,mean,std,min,50%,90%,99%,99.9%,max
TrafficDirection,,,,,,,,,
Downlink,64945.0,45.647182,1547.513329,0.00106,1.491803,39.952438,158.783077,5476.235519,104022.000000
Uplink,64875.0,3.294869,96.105543,0.00124,0.340776,2.443647,20.766784,302.234675,8439.682444


<a id="traffic-by-operator"></a>
### Total traffic by Operator and Direction (GB)

In [17]:
(traffic_df.pivot_table(index='RadioOperatorName',
                        columns='TrafficDirection',
                        values='TrafficVolume',
                        aggfunc='sum') / 1024).round(2)

TrafficDirection,Downlink,Uplink
RadioOperatorName,,
Operator A,691.15,63.43
Operator B,1869.89,104.94
Operator C,334.03,40.38


<a id="traffic-by-hour"></a>
### Traffic by Hour by Direction

In [18]:
traffic_df['Date'] = traffic_df['Timestamp'].dt.date
traffic_df['Hour'] = traffic_df['Timestamp'].dt.hour

In [19]:
traffic_df.pivot_table(index='Hour',
                       columns='TrafficDirection',
                       values='TrafficVolume',
                       aggfunc='sum').round(0)

TrafficDirection,Downlink,Uplink
Hour,,
0,150792.0,11772.0
1,133788.0,10577.0
2,31959.0,1714.0
3,27504.0,4577.0
4,11902.0,1761.0
5,10614.0,1450.0
6,23508.0,2248.0
7,42947.0,4088.0
8,65516.0,11824.0


<a id="saving"></a>
### Saving the cleaned data
Parquet keeps the dtypes and the timezone, and reloads far faster than the CSV.

In [20]:
traffic_df.to_parquet('../traffic_clean.parquet', index=False)
traffic_df.shape

(129820, 11)